In [1]:
import os
os.chdir("..")

In [2]:
import sys
sys.path.append(r"D:\Work\Projects\sam3")

In [4]:
import torch

device = torch.device("cpu")

In [5]:
import cv2
import numpy as np
from ultralytics import YOLO
from scipy.interpolate import splprep, splev

from collections import defaultdict

In [7]:
def create_walking_corridor(
        frame,
        width_ratio=0.35,
        height_ratio=0.75
):
    """
    Creates a trapezoid representing walking path.
    """

    h, w = frame.shape[:2]

    # bottom center = user's feet direction
    bottom_y = h
    top_y = int(h * (1-height_ratio))

    center_x = w // 2

    bottom_half = int(w * width_ratio)
    top_half = int(bottom_half * 0.25)

    points = np.array([
        [center_x-top_half, top_y],
        [center_x+top_half, top_y],
        [center_x+bottom_half, bottom_y],
        [center_x-bottom_half, bottom_y]
    ])

    mask = np.zeros((h,w), dtype=np.uint8)

    cv2.fillPoly(
        mask,
        [points],
        255
    )

    return mask, points


frame = cv2.imread("samples/footpath.jpg")

mask, pts = create_walking_corridor(frame)

overlay = frame.copy()
overlay[mask==255] = (0,255,0)

out = cv2.addWeighted(
    frame,
    0.7,
    overlay,
    0.3,
    0
)

cv2.imwrite(
    "corridor.png",
    out
)

True

In [3]:
from sam3 import build_sam3_image_model


predictor = build_sam3_image_model(
    checkpoint_path="sam3.pt",
    device='cpu'
)


obstacle_prompts = [
    "person",
    "chair",
    "bicycle",
    "car",
    "pole",
    "tree",
    "box",
    "animal"
]


def detect_obstacles(frame):

    results = []

    for prompt in obstacle_prompts:

        masks = predictor.predict(
            image=frame,
            text_prompt=prompt
        )

        for mask in masks:
            results.append({
                "type": prompt,
                "mask": mask
            })

    return results

D:\Work\venv\lib\site-packages\torch\amp\autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
D:\Work\venv\lib\site-packages\timm\models\layers\__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


FileNotFoundError: [Errno 2] No such file or directory: 'sam3.pt'

In [1]:
import sys
sys.path.append(r"D:\Work\Projects\sam3")

In [2]:
import os
import sam3
import torch

sam3_root = os.path.join(os.path.dirname(sam3.__file__), "..")

# use all available GPUs on the machine
gpus_to_use = range(torch.cuda.device_count())
# # use only a single GPU
# gpus_to_use = [torch.cuda.current_device()]

D:\Work\venv\lib\site-packages\torch\amp\autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(
D:\Work\venv\lib\site-packages\timm\models\layers\__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [5]:
from sam3.model_builder import build_sam3_video_predictor

predictor = build_sam3_video_predictor(gpus_to_use=gpus_to_use)

INFO 2026-06-24 01:18:46,845 24312 sam3_video_predictor.py: 109: using the following GPU IDs: []


AssertionError: 